## Nacimientos, matrimonios y defunciones 2020 y 2010

En este notebook procesaremos, para cada año, 52 JSON, uno por cada provincia y ciudad autónoma.  
Cada uno de ellos contiene el total de nacimientos, matrimonios y defunciones por municipios en 2020 y 2010. 

In [1]:
import pandas as pd
import numpy as np
import requests

Para extraer tanto los datos de 2020 como de 2010, utilizaremos la siguiente función que tiene como argumento una lista con los números de cada una de las tablas de las provincias y ciudades autónomas:

In [2]:
def ine_datos(CodPro):
    
    PobNacList = list()
    NacList = list()
    PobMatList = list()
    MatList = list()
    PobDefList = list()
    DefList = list()
    
    for codigo in CodPro:
        path = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/{cod_serie}?tip=AM&'
        path = path.format(cod_serie=codigo)
        json_request = requests.get(path).json()
        for dato in json_request: 
            for i in range(0,len(dato['Data'])):
                if 'nacidos vivos por residencia materna' in dato['Nombre']:
                    PobNacList.append(dato['Nombre'])
                    NacList.append(dato['Data'][i]['Valor'])
                elif 'matrimonios por el lugar en que han fijado residencia' in dato['Nombre']:
                    PobMatList.append(dato['Nombre'])
                    MatList.append(dato['Data'][i]['Valor'])
                elif 'fallecidos por el lugar de residencia' in dato['Nombre']:
                    PobDefList.append(dato['Nombre'])
                    DefList.append(dato['Data'][i]['Valor']) 
    
    Nacimientos = pd.DataFrame({
    'Codigo Municipio' : PobNacList,
    'Nacimientos': NacList
    })

    Matrimonios = pd.DataFrame({
    'Codigo Municipio' : PobMatList,
    'Matrimonios': MatList
    })

    Defunciones = pd.DataFrame({
    'Codigo Municipio' : PobMatList,
    'Defunciones': DefList
    }) 
    
    return Nacimientos, Matrimonios, Defunciones

### Nacimientos, matrimonios y defunciones 2020
 
Obtendremos tres DataFrame con la siguiente información común:
- Y2020
- Nombre Provincia
- Codigo Provincia
- Nombre Municipio
- Codigo Municipio 

Y con cada uno de ellos con uno de los siguientes campos:
- Nacimientos
- Matrimonios
- Defunciones

Codigos de las tablas correspondientes a 2020:

In [3]:
CodPro20 = list(range(50370, 50421+1))

In [4]:
Nacimientos, Matrimonios, Defunciones = ine_datos(CodPro20)

In [5]:
Nacimientos.head()

,Codigo Municipio,Nacimientos
0,"01051 Agurain/Salvatierra, nacidos vivos por r...",45.0
1,"01001 Alegría-Dulantzi, nacidos vivos por resi...",17.0
2,"01002 Amurrio, nacidos vivos por residencia ma...",85.0
3,"01049 Añana, nacidos vivos por residencia materna",0.0
4,"01003 Aramaio, nacidos vivos por residencia ma...",5.0


In [6]:
Matrimonios.head()

,Codigo Municipio,Matrimonios
0,"01051 Agurain/Salvatierra, matrimonios por el ...",7.0
1,"01001 Alegría-Dulantzi, matrimonios por el lug...",2.0
2,"01002 Amurrio, matrimonios por el lugar en que...",22.0
3,"01049 Añana, matrimonios por el lugar en que h...",0.0
4,"01003 Aramaio, matrimonios por el lugar en que...",4.0


In [7]:
Defunciones.head()

,Codigo Municipio,Defunciones
0,"01051 Agurain/Salvatierra, matrimonios por el ...",50.0
1,"01001 Alegría-Dulantzi, matrimonios por el lug...",11.0
2,"01002 Amurrio, matrimonios por el lugar en que...",98.0
3,"01049 Añana, matrimonios por el lugar en que h...",2.0
4,"01003 Aramaio, matrimonios por el lugar en que...",18.0


Cargamos la tabla 01_Output_ProvMun_20.xlsx para posteriormente realizar el merge con las tablas anteriores:

In [8]:
ProvMun = pd.read_excel('01_Output_ProvMun_20.xlsx', dtype='object') 
ProvMun.head()

,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio
0,Araba/Álava,01,Agurain/Salvatierra,01051
1,Araba/Álava,01,Alegría-Dulantzi,01001
2,Araba/Álava,01,Amurrio,01002
3,Araba/Álava,01,Añana,01049
4,Araba/Álava,01,Aramaio,01003


Incluimos en las tablas Nacimientos, Matrimonios y Defunciones el campo Codigo Municipio:

In [9]:
MunList = ProvMun['Codigo Municipio'].tolist()

In [10]:
def check_list(x):
    for l in MunList:
        if l in x:
            return l
    return ''

In [11]:
Nacimientos['Codigo Municipio'] = Nacimientos['Codigo Municipio'].map(lambda x: check_list(x))
Matrimonios['Codigo Municipio'] = Matrimonios['Codigo Municipio'].map(lambda x: check_list(x))
Defunciones['Codigo Municipio'] = Defunciones['Codigo Municipio'].map(lambda x: check_list(x))

Realizamos el merge con la tabla ProvMun:

In [12]:
Nacimientos20 = Nacimientos.merge(ProvMun, on='Codigo Municipio')
Matrimonios20 = Matrimonios.merge(ProvMun, on='Codigo Municipio')
Defunciones20 = Defunciones.merge(ProvMun, on='Codigo Municipio')

In [13]:
Nacimientos20['Y2020'] = 2020
ColNue = ['Y2020', 'Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Nacimientos']
Nacimientos20= Nacimientos20[ColNue]
Nacimientos20.head()

,Y2020,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Nacimientos
0,2020,Araba/Álava,01,Agurain/Salvatierra,01051,45.0
1,2020,Araba/Álava,01,Alegría-Dulantzi,01001,17.0
2,2020,Araba/Álava,01,Amurrio,01002,85.0
3,2020,Araba/Álava,01,Añana,01049,0.0
4,2020,Araba/Álava,01,Aramaio,01003,5.0


In [14]:
Matrimonios20['Y2020'] = 2020
ColNue = ['Y2020','Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Matrimonios']
Matrimonios20= Matrimonios20[ColNue]
Matrimonios20.head()

,Y2020,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Matrimonios
0,2020,Araba/Álava,01,Agurain/Salvatierra,01051,7.0
1,2020,Araba/Álava,01,Alegría-Dulantzi,01001,2.0
2,2020,Araba/Álava,01,Amurrio,01002,22.0
3,2020,Araba/Álava,01,Añana,01049,0.0
4,2020,Araba/Álava,01,Aramaio,01003,4.0


In [15]:
Defunciones20['Y2020'] = 2020
ColNue = ['Y2020','Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Defunciones']
Defunciones20= Defunciones20[ColNue]
Defunciones20.head()

,Y2020,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Defunciones
0,2020,Araba/Álava,01,Agurain/Salvatierra,01051,50.0
1,2020,Araba/Álava,01,Alegría-Dulantzi,01001,11.0
2,2020,Araba/Álava,01,Amurrio,01002,98.0
3,2020,Araba/Álava,01,Añana,01049,2.0
4,2020,Araba/Álava,01,Aramaio,01003,18.0


In [16]:
Nacimientos20.to_excel('05_Output_Nacimientos_20.xlsx', header = True, index = False)
Matrimonios20.to_excel('05_Output_Matrimonios_20.xlsx', header = True, index = False)
Defunciones20.to_excel('05_Output_Defunciones_20.xlsx', header = True, index = False)

### Nacimientos, matrimonios y defunciones 2010
 
Obtendremos tres DataFrame con la siguiente información común:
- Y2010
- Nombre Provincia
- Codigo Provincia
- Nombre Municipio
- Codigo Municipio 

Y con cada uno de ellos con uno de los siguientes campos:
- Nacimientos
- Matrimonios
- Defunciones

Codigos de las tablas correspondientes a 2010:

In [17]:
CodPro10 = list(range(22863, 22914+1))

In [18]:
Nacimientos, Matrimonios, Defunciones = ine_datos(CodPro10)

In [19]:
Nacimientos.head()

,Codigo Municipio,Nacimientos
0,"02001 Abengibre, nacidos vivos por residencia ...",1.0
1,"02002 Alatoz, nacidos vivos por residencia mat...",2.0
2,"02003 Albacete, nacidos vivos por residencia m...",1986.0
3,"02004 Albatana, nacidos vivos por residencia m...",5.0
4,"02005 Alborea, nacidos vivos por residencia ma...",8.0


In [20]:
Matrimonios.head()

,Codigo Municipio,Matrimonios
0,"02001 Abengibre, matrimonios por el lugar en q...",1.0
1,"02002 Alatoz, matrimonios por el lugar en que ...",1.0
2,"02003 Albacete, matrimonios por el lugar en qu...",683.0
3,"02004 Albatana, matrimonios por el lugar en qu...",3.0
4,"02005 Alborea, matrimonios por el lugar en que...",2.0


In [21]:
Defunciones.head()

,Codigo Municipio,Defunciones
0,"02001 Abengibre, matrimonios por el lugar en q...",13.0
1,"02002 Alatoz, matrimonios por el lugar en que ...",12.0
2,"02003 Albacete, matrimonios por el lugar en qu...",1096.0
3,"02004 Albatana, matrimonios por el lugar en qu...",10.0
4,"02005 Alborea, matrimonios por el lugar en que...",11.0


Cargamos la tabla 01_Output_ProvMun_10.xlsx para posteriormente realizar el merge con las tablas anteriores:

In [22]:
ProvMun = pd.read_excel('01_Output_ProvMun_10.xlsx', dtype='object') 
ProvMun.head()

,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio
0,Araba/Álava,01,Agurain/Salvatierra,01051
1,Araba/Álava,01,Alegría-Dulantzi,01001
2,Araba/Álava,01,Amurrio,01002
3,Araba/Álava,01,Añana,01049
4,Araba/Álava,01,Aramaio,01003


In [23]:
MunList = ProvMun['Codigo Municipio'].tolist()

In [24]:
def check_list(x):
    for l in MunList:
        if l in x:
            return l
    return ''

In [25]:
Nacimientos['Codigo Municipio'] = Nacimientos['Codigo Municipio'].map(lambda x: check_list(x))
Matrimonios['Codigo Municipio'] = Matrimonios['Codigo Municipio'].map(lambda x: check_list(x))
Defunciones['Codigo Municipio'] = Defunciones['Codigo Municipio'].map(lambda x: check_list(x))

Realizamos el merge con la tabla ProvMun:

In [26]:
Nacimientos10 = Nacimientos.merge(ProvMun, on='Codigo Municipio')
Matrimonios10 = Matrimonios.merge(ProvMun, on='Codigo Municipio')
Defunciones10 = Defunciones.merge(ProvMun, on='Codigo Municipio')

In [27]:
Nacimientos10['Y2010'] = 2010
ColNue = ['Y2010', 'Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Nacimientos']
Nacimientos10= Nacimientos10[ColNue]
Nacimientos10.head()

,Y2010,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Nacimientos
0,2010,Albacete,02,Abengibre,02001,1.0
1,2010,Albacete,02,Alatoz,02002,2.0
2,2010,Albacete,02,Albacete,02003,1986.0
3,2010,Albacete,02,Albatana,02004,5.0
4,2010,Albacete,02,Alborea,02005,8.0


In [28]:
Matrimonios10['Y2010'] = 2010
ColNue = ['Y2010','Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Matrimonios']
Matrimonios10= Matrimonios10[ColNue]
Matrimonios10.head()

,Y2010,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Matrimonios
0,2010,Albacete,02,Abengibre,02001,1.0
1,2010,Albacete,02,Alatoz,02002,1.0
2,2010,Albacete,02,Albacete,02003,683.0
3,2010,Albacete,02,Albatana,02004,3.0
4,2010,Albacete,02,Alborea,02005,2.0


In [29]:
Defunciones10['Y2010'] = 2010
ColNue = ['Y2010','Nombre Provincia', 'Codigo Provincia', 'Nombre Municipio','Codigo Municipio', 'Defunciones']
Defunciones10= Defunciones10[ColNue]
Defunciones10.head()

,Y2010,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Defunciones
0,2010,Albacete,02,Abengibre,02001,13.0
1,2010,Albacete,02,Alatoz,02002,12.0
2,2010,Albacete,02,Albacete,02003,1096.0
3,2010,Albacete,02,Albatana,02004,10.0
4,2010,Albacete,02,Alborea,02005,11.0


In [30]:
Nacimientos10.to_excel('05_Output_Nacimientos_10.xlsx', header = True, index = False)
Matrimonios10.to_excel('05_Output_Matrimonios_10.xlsx', header = True, index = False)
Defunciones10.to_excel('05_Output_Defunciones_10.xlsx', header = True, index = False)